# Adults Income Dataset

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import StratifiedKFold, learning_curve, cross_validate
from sklearn.metrics import confusion_matrix


SEED=49
np.random.seed(SEED)
random.seed(SEED)

### Data Loading

We load the Adult Income dataset and inspect its basic structure.

In [3]:
DATA_PATH = "../adult.csv"
df = pd.read_csv(DATA_PATH)

df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25.0,Private,226802.0,11th,7.0,Never-married,Machine-op-inspct,Own-child,Black,Male,0.0,0.0,40.0,United-States,<=50K
1,38.0,Private,89814.0,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K
2,28.0,Local-gov,336951.0,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,White,Male,0.0,0.0,40.0,United-States,>50K
3,44.0,Private,160323.0,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688.0,0.0,40.0,United-States,>50K
4,34.0,Private,198693.0,10th,6.0,Never-married,Other-service,Not-in-family,White,Male,0.0,0.0,30.0,United-States,<=50K


If we look closely, `education` and `education-num` are redundant because they contain the same information, expressed in two different ways. For the purposes of creating the model, it would be best to omit one of them. We have decided to omit `education`, since `education-num` is a variable that is already numerical, which although it refers to categories, since there are many categories it will be better to treat it as numerical. 

The feature `fnlwgt` represents a census sampling weight rather than an individual-level attribute. Since our goal is predictive modeling at the individual level, this variable does not carry meaningful explanatory information and is excluded from the feature set.

### Target Variable Distribution

We analyze class balance to motivate metric selection.

In [4]:
target_col = "class"
df[target_col].value_counts()

<=50K    34014
>50K     11208
Name: class, dtype: int64

Because the adults dataset is unbalanced, accuracy may overestimate performance in the majority class. Therefore, we adopt F1 as the primary evaluation metric.

## Experiments

A useful working hypothesis for this dataset is that:

- Because Adult becomes high-dimensional and sparse after one-hot encoding, PCA and RP may compress redundant directions efficiently
- ICA may produce less stable components than PCA because the feature space mixes standardized numerical variables with many sparse indicator columns
- GMM will likely need stronger structural assumptions than K-Means in the raw feature space, so diagonal covariance is a reasonable starting point
- Cluster features may add some coarse socioeconomic grouping signal, but the original supervised representation may still remain stronger than cluster-only inputs

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

X = df.drop(columns=["class", "fnlwgt", "education"])
y = df["class"]

num_cols = ["age", "hours-per-week", "capital-gain", "capital-loss", 'education-num']
cat_cols = ['workclass','marital-status','occupation','relationship','race','sex','native-country']

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)


70% training set, 20% testing set, 10% validation set

In [6]:
from sklearn.model_selection import train_test_split

y_bin = (pd.Series(y).astype(str).str.strip() == ">50K").astype(int).values

TEST_SIZE = 0.20
VAL_SIZE  = 0.10

X_train_t, X_test, y_train_t, y_test = train_test_split(
    X, y_bin, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

val_size_within_train = VAL_SIZE / (1.0 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_t, y_train_t, test_size=val_size_within_train, stratify=y_train_t, random_state=SEED
)

In [7]:
from scipy import sparse

def ensure_dense(X):
    """
    Convert sparse matrices to dense numpy arrays when needed
    
    """
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

In [8]:
X_train_sp = preprocessor.fit_transform(X_train)
X_val_sp = preprocessor.transform(X_val)
X_test_sp = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
input_dim = len(feature_names)

X_train_dense = ensure_dense(X_train_sp).astype(np.float64, copy=False)
X_val_dense = ensure_dense(X_val_sp).astype(np.float64, copy=False)
X_test_dense = ensure_dense(X_test_sp).astype(np.float64, copy=False)

# For unsupervised analyses that use the whole dataset, transform the full feature set
X_all_sp = preprocessor.transform(X)
X_all = ensure_dense(X_all_sp).astype(np.float64, copy=False)
y_all = y.copy()

print("Encoded input dimension:", input_dim)
print("All-data matrix shape:", X_all.shape)

Encoded input dimension: 87
All-data matrix shape: (45222, 87)


## Step 1 (Clustering on raw data)

In [11]:
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits
from sklearn.metrics import adjusted_rand_score,confusion_matrix,f1_score,normalized_mutual_info_score,silhouette_score
from sklearn.mixture import GaussianMixture

# Search grids for Step 1
KMEANS_GRID = [2, 3, 4, 5, 6, 8, 10]
GMM_GRID = [2, 3, 4, 5, 6, 8, 10]

def evaluate_clustering(y_true, labels, X_eval=None):
    """
    Compute internal and external clustering diagnostics
    """
    out = {}
    unique_labels = np.unique(labels)
    if X_eval is not None and len(unique_labels) > 1 and len(unique_labels) < len(labels):
        out["silhouette"] = float(silhouette_score(X_eval, labels))
    else:
        out["silhouette"] = np.nan
    out["ari"] = float(adjusted_rand_score(y_true, labels))
    out["nmi"] = float(normalized_mutual_info_score(y_true, labels))
    return out

def clustering_stability_kmeans(X, n_clusters, n_runs=5, base_seed=SEED, **kwargs):
    """
    Average pairwise ARI across repeated K-Means runs as a stability proxy
    """
    labels_runs = []
    for i in range(n_runs):
        km = KMeans(n_clusters=n_clusters, random_state=base_seed + i, **kwargs)
        labels_runs.append(km.fit_predict(X))
    pairwise = []
    for i in range(len(labels_runs)):
        for j in range(i + 1, len(labels_runs)):
            pairwise.append(adjusted_rand_score(labels_runs[i], labels_runs[j]))
    return float(np.mean(pairwise)) if pairwise else np.nan

def run_kmeans_grid(X_mat, y_true, cluster_grid=KMEANS_GRID, random_state=SEED, n_init=20, max_iter=500):
    rows = []
    for k in cluster_grid:
        km = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=n_init,
            max_iter=max_iter,
        )
        labels = km.fit_predict(X_mat)
        metrics = evaluate_clustering(y_true, labels, X_eval=X_mat)
        rows.append(
            {
                "model": "kmeans",
                "n_clusters": k,
                "inertia": float(km.inertia_),
                "stability_ari": clustering_stability_kmeans(
                    X_mat,
                    n_clusters=k,
                    n_runs=5,
                    base_seed=random_state,
                    n_init=n_init,
                    max_iter=max_iter,
                ),
                **metrics,
            }
        )
    return pd.DataFrame(rows)


def run_gmm_grid(X_mat, y_true, component_grid=GMM_GRID, random_state=SEED, covariance_type="diag", n_init=3, max_iter=300):
    rows = []
    fitted = {}
    for k in component_grid:
        gmm = GaussianMixture(
            n_components=k,
            covariance_type=covariance_type,
            n_init=n_init,
            max_iter=max_iter,
            random_state=random_state,
            reg_covar=1e-3,
        )
        gmm.fit(X_mat)
        labels = gmm.predict(X_mat)
        metrics = evaluate_clustering(y_true, labels, X_eval=X_mat)
        rows.append(
            {
                "model": "gmm",
                "n_components": k,
                "covariance_type": covariance_type,
                "bic": float(gmm.bic(X_mat)),
                "aic": float(gmm.aic(X_mat)),
                **metrics,
            }
        )
        fitted[k] = gmm
    return pd.DataFrame(rows), fitted


In [ ]:
kmeans_raw_results = run_kmeans_grid(X_all, y_all)

In [ ]:
import matplotlib.pyplot as plt

def plot_metric_curve(df, x, y, title, ylabel=None):
    tmp = df.sort_values(x)
    plt.figure()
    plt.plot(tmp[x], tmp[y], marker="o")
    plt.title(title)
    plt.xlabel(x)
    plt.ylabel(ylabel or y)
    plt.show()

In [ ]:
from IPython.display import display

display(kmeans_raw_results)

plot_metric_curve(kmeans_raw_results, x="n_clusters", y="silhouette", title="Adult raw data - K-Means silhouette")
plot_metric_curve(kmeans_raw_results, x="n_clusters", y="stability_ari", title="Adult raw data - K-Means stability")
plot_metric_curve(kmeans_raw_results, x="n_clusters", y="inertia", title="Adult raw data - K-Means inertia")

In [ ]:
gmm_raw_results, gmm_raw_models = run_gmm_grid(X_all, y_all, covariance_type="diag")
display(gmm_raw_results)

plot_metric_curve(gmm_raw_results, x="n_components", y="silhouette", title="Adult raw data - GMM silhouette")
plot_metric_curve(gmm_raw_results, x="n_components", y="bic", title="Adult raw data - GMM BIC")
plot_metric_curve(gmm_raw_results, x="n_components", y="aic", title="Adult raw data - GMM AIC")

In [ ]:
#Select baseline raw clustering models
BEST_KMEANS_K = int(
    kmeans_raw_results.sort_values(["silhouette", "stability_ari"], ascending=[False, False]).iloc[0]["n_clusters"]
)

BEST_GMM_K = int(
    gmm_raw_results.sort_values(["bic", "aic"], ascending=[True, True]).iloc[0]["n_components"]
)

kmeans_raw_best = KMeans(
    n_clusters=BEST_KMEANS_K,
    random_state=SEED,
    n_init=20,
    max_iter=500,
)
kmeans_raw_labels = kmeans_raw_best.fit_predict(X_all)

gmm_raw_best = gmm_raw_models[BEST_GMM_K]
gmm_raw_labels = gmm_raw_best.predict(X_all)

print("Selected raw K-Means k:", BEST_KMEANS_K)
print("Selected raw GMM components:", BEST_GMM_K)

raw_summary = pd.DataFrame(
    [
        {"model": "kmeans_raw_best", "chosen_k": BEST_KMEANS_K, **evaluate_clustering(y_all, kmeans_raw_labels, X_all)},
        {"model": "gmm_raw_best", "chosen_k": BEST_GMM_K, **evaluate_clustering(y_all, gmm_raw_labels, X_all)},
    ]
)
display(raw_summary)


## Step 2 (Dimensionality reduction on raw data)

In [ ]:
MAX_DIM = min(30, X_all.shape[1])
DIM_GRID = sorted(set([2, 3, 5, 8, 10, 15, 20, 25, 30]))
DIM_GRID = [d for d in DIM_GRID if d <= MAX_DIM]
print("Dimension grid:", DIM_GRID)

In [ ]:
from sklearn.decomposition import FastICA, PCA

In [ ]:
def pca_diagnostics(X_mat, dim_grid=DIM_GRID, random_state=SEED):
    pca_full = PCA(random_state=random_state)
    pca_full.fit(X_mat)

    cum_var = np.cumsum(pca_full.explained_variance_ratio_)
    pca_rows = []
    for d in dim_grid:
        pca = PCA(n_components=d, random_state=random_state)
        Z = pca.fit_transform(X_mat)
        recon = pca.inverse_transform(Z)
        mse = float(np.mean((X_mat - recon) ** 2))
        pca_rows.append(
            {
                "method": "pca",
                "n_components": d,
                "explained_variance_ratio_sum": float(np.sum(pca.explained_variance_ratio_)),
                "reconstruction_mse": mse,
            }
        )
    return pca_full, pd.DataFrame(pca_rows), cum_var

pca_full, pca_results, pca_cum_var = pca_diagnostics(X_all)
display(pca_results)

plt.figure()
plt.plot(np.arange(1, len(pca_cum_var) + 1), pca_cum_var, marker="o")
plt.axhline(0.95, linestyle="--")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("Adult - PCA cumulative explained variance")
plt.show()

In [ ]:
from scipy import stats
def ica_diagnostics(X_mat, dim_grid=DIM_GRID, random_state=SEED):
    rows = []
    models = {}
    for d in dim_grid:
        ica = FastICA(
            n_components=d,
            random_state=random_state,
            whiten="unit-variance",
            max_iter=1000,
            tol=1e-4,
        )
        Z = ica.fit_transform(X_mat)
        component_kurtosis = np.mean(np.abs(stats.kurtosis(Z, axis=0, fisher=True, bias=False)))
        if hasattr(ica, "inverse_transform"):
            recon = ica.inverse_transform(Z)
            mse = float(np.mean((X_mat - recon) ** 2))
        else:
            mse = np.nan
        rows.append(
            {
                "method": "ica",
                "n_components": d,
                "mean_abs_kurtosis": float(component_kurtosis),
                "reconstruction_mse": mse,
                "n_iter": int(getattr(ica, "n_iter_", -1)),
            }
        )
        models[d] = ica
    return pd.DataFrame(rows), models

ica_results, ica_models = ica_diagnostics(X_all)
display(ica_results)

plot_metric_curve(ica_results, x="n_components", y="mean_abs_kurtosis", title="Adult - ICA mean |kurtosis|", ylabel="mean |kurtosis|")

In [ ]:
from sklearn.random_projection import GaussianRandomProjection

def rp_diagnostics(X_mat, dim_grid=DIM_GRID, seeds=(49, 35, 11, 7, 21)):
    rows = []
    best_models = {}
    for d in dim_grid:
        seed_rows = []
        candidate_models = []
        for s in seeds:
            rp = GaussianRandomProjection(n_components=d, random_state=s)
            Z = rp.fit_transform(X_mat)
            if hasattr(rp, "inverse_transform"):
                recon = rp.inverse_transform(Z)
                mse = float(np.mean((X_mat - recon) ** 2))
            else:
                mse = np.nan
            sample_var = float(np.mean(np.var(Z, axis=0)))
            seed_rows.append({"seed": s, "reconstruction_mse": mse, "avg_component_var": sample_var})
            candidate_models.append((s, rp, mse))
        mse_values = [r["reconstruction_mse"] for r in seed_rows]
        rows.append(
            {
                "method": "rp",
                "n_components": d,
                "mean_reconstruction_mse": float(np.mean(mse_values)),
                "std_reconstruction_mse": float(np.std(mse_values)),
                "mean_component_var": float(np.mean([r["avg_component_var"] for r in seed_rows])),
            }
        )
        best_seed, best_rp, _ = sorted(candidate_models, key=lambda x: x[2])[0]
        best_models[d] = {"seed": best_seed, "model": best_rp}
    return pd.DataFrame(rows), best_models

rp_results, rp_models = rp_diagnostics(X_all)
display(rp_results)

plot_metric_curve(rp_results, x="n_components", y="mean_reconstruction_mse", title="Adult - RP mean reconstruction MSE")
plot_metric_curve(rp_results, x="n_components", y="std_reconstruction_mse", title="Adult - RP reconstruction variability")

In [ ]:
PCA_DIM = int(pca_results[pca_results["explained_variance_ratio_sum"] >= 0.95]["n_components"].min())     if (pca_results["explained_variance_ratio_sum"] >= 0.95).any() else int(pca_results.sort_values("explained_variance_ratio_sum", ascending=False).iloc[0]["n_components"])

ICA_DIM = int(ica_results.sort_values(["mean_abs_kurtosis", "reconstruction_mse"], ascending=[False, True]).iloc[0]["n_components"])
RP_DIM = int(rp_results.sort_values(["mean_reconstruction_mse", "std_reconstruction_mse"], ascending=[True, True]).iloc[0]["n_components"])

print("Selected PCA dim:", PCA_DIM)
print("Selected ICA dim:", ICA_DIM)
print("Selected RP dim:", RP_DIM)

In [ ]:
pca_selected = PCA(n_components=PCA_DIM, random_state=SEED)
X_all_pca = pca_selected.fit_transform(X_all)

ica_selected = FastICA(
    n_components=ICA_DIM,
    random_state=SEED,
    whiten="unit-variance",
    max_iter=1000,
    tol=1e-4,
)
X_all_ica = ica_selected.fit_transform(X_all)

rp_selected = rp_models[RP_DIM]["model"]
X_all_rp = rp_selected.transform(X_all)

print("PCA shape:", X_all_pca.shape)
print("ICA shape:", X_all_ica.shape)
print("RP shape:", X_all_rp.shape)


These plots do not replace the quantitative comparisons, but gives us a quick look of the first 2 components

In [ ]:
def plot_first_two_components(Z, y_true, title):
    if Z.shape[1] < 2:
        print(f"{title}: not enough components to plot.")
        return
    
    y_codes = pd.Series(y_true).astype("category").cat.codes

    plt.figure(figsize=(7, 5))
    plt.scatter(Z[:, 0], Z[:, 1], c=y_codes, s=8, alpha=0.5)
    plt.title(title)
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.tight_layout()
    plt.show()

plot_first_two_components(X_all_pca, y_all, "Adult - PCA first two components")
plot_first_two_components(X_all_ica, y_all, "Adult - ICA first two components")
plot_first_two_components(X_all_rp, y_all, "Adult - RP first two components")


## Step 3 (Clustering in reduced-dimensional spaces)

In [ ]:
reduced_spaces = {
    "pca": X_all_pca,
    "ica": X_all_ica,
    "rp": X_all_rp,
}

selected_dims = {
    "pca": PCA_DIM,
    "ica": ICA_DIM,
    "rp": RP_DIM,
}

def evaluate_reduced_space_clustering(space_name, X_red, y_true):
    km_df = run_kmeans_grid(X_red, y_true)
    gmm_df, _ = run_gmm_grid(X_red, y_true, covariance_type="diag")

    km_best = km_df.sort_values(["silhouette", "stability_ari"], ascending=[False, False]).iloc[0]
    gmm_best = gmm_df.sort_values(["bic", "aic"], ascending=[True, True]).iloc[0]

    return pd.DataFrame(
        [
            {
                "space": space_name,
                "dim": selected_dims[space_name],
                "cluster_model": "kmeans",
                "selected_k": int(km_best["n_clusters"]),
                "silhouette": float(km_best["silhouette"]),
                "ari": float(km_best["ari"]),
                "nmi": float(km_best["nmi"]),
                "aux_metric_1": float(km_best["inertia"]),
                "aux_metric_2": float(km_best["stability_ari"]),
            },
            {
                "space": space_name,
                "dim": selected_dims[space_name],
                "cluster_model": "gmm",
                "selected_k": int(gmm_best["n_components"]),
                "silhouette": float(gmm_best["silhouette"]),
                "ari": float(gmm_best["ari"]),
                "nmi": float(gmm_best["nmi"]),
                "aux_metric_1": float(gmm_best["bic"]),
                "aux_metric_2": float(gmm_best["aic"]),
            },
        ]
    )

step3_results = pd.concat(
    [evaluate_reduced_space_clustering(name, X_red, y_all) for name, X_red in reduced_spaces.items()],
    ignore_index=True,
)

display(step3_results.sort_values(["cluster_model", "silhouette"], ascending=[True, False]))

In [ ]:
comparison_rows = []

raw_kmeans_best_row = kmeans_raw_results.sort_values(["silhouette", "stability_ari"], ascending=[False, False]).iloc[0]
raw_gmm_best_row = gmm_raw_results.sort_values(["bic", "aic"], ascending=[True, True]).iloc[0]

comparison_rows.append(
    {
        "space": "raw",
        "dim": X_all.shape[1],
        "cluster_model": "kmeans",
        "selected_k": int(raw_kmeans_best_row["n_clusters"]),
        "silhouette": float(raw_kmeans_best_row["silhouette"]),
        "ari": float(raw_kmeans_best_row["ari"]),
        "nmi": float(raw_kmeans_best_row["nmi"]),
    }
)

comparison_rows.append(
    {
        "space": "raw",
        "dim": X_all.shape[1],
        "cluster_model": "gmm",
        "selected_k": int(raw_gmm_best_row["n_components"]),
        "silhouette": float(raw_gmm_best_row["silhouette"]),
        "ari": float(raw_gmm_best_row["ari"]),
        "nmi": float(raw_gmm_best_row["nmi"]),
    }
)

comparison_rows.extend(step3_results[["space", "dim", "cluster_model", "selected_k", "silhouette", "ari", "nmi"]].to_dict("records"))

cluster_comparison_table = pd.DataFrame(comparison_rows)
display(cluster_comparison_table.sort_values(["cluster_model", "silhouette"], ascending=[True, False]))

## Step 4 (Neural networks on dimensionality-reduced data)

In [ ]:
import torch
import torch.nn as nn

class MLP_tmp(nn.Module):
    def __init__(self, d, h=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(d, h),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(h, 2)

    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import copy

device = "cuda" if torch.cuda.is_available() else "cpu"

def make_binary_loaders(X_train_np, y_train_np, X_val_np, y_val_np, X_test_np, y_test_np, batch_size=64, seed=SEED):
    X_train_t = torch.tensor(np.asarray(X_train_np), dtype=torch.float32)
    X_val_t = torch.tensor(np.asarray(X_val_np), dtype=torch.float32)
    X_test_t = torch.tensor(np.asarray(X_test_np), dtype=torch.float32)

    y_train_t = torch.tensor(np.asarray(y_train_np), dtype=torch.long)
    y_val_t = torch.tensor(np.asarray(y_val_np), dtype=torch.long)
    y_test_t = torch.tensor(np.asarray(y_test_np), dtype=torch.long)

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)
    test_ds = TensorDataset(X_test_t, y_test_t)

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

@torch.no_grad()
def evaluate_binary_model(model, loader, device=device):
    model.eval()
    preds, ys = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        ys.append(yb.cpu().numpy())
    y_pred = np.concatenate(preds)
    y_true = np.concatenate(ys)
    return {
        "f1": float(f1_score(y_true, y_pred)),
        "y_pred": y_pred,
        "y_true": y_true,
        "cm": confusion_matrix(y_true, y_pred),
    }

def train_mlp_binary(X_train_np, y_train_np, X_val_np, y_val_np, X_test_np, y_test_np,
                     hidden_dim=128, lr=1e-3, batch_size=64, max_epochs=60, patience=8, seed=SEED):
    set_global_seed(seed)
    train_loader, val_loader, test_loader = make_binary_loaders(
        X_train_np, y_train_np, X_val_np, y_val_np, X_test_np, y_test_np, batch_size=batch_size, seed=seed
    )

    model = MLP_tmp(d=X_train_np.shape[1], h=hidden_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = np.inf
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_losses.append(loss.item())

        val_eval = evaluate_binary_model(model, val_loader, device=device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "val_loss": float(np.mean(val_losses)),
                "val_f1": val_eval["f1"],
            }
        )

        if np.mean(val_losses) < best_val_loss - 1e-5:
            best_val_loss = np.mean(val_losses)
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    val_eval = evaluate_binary_model(model, val_loader, device=device)
    test_eval = evaluate_binary_model(model, test_loader, device=device)

    return model, pd.DataFrame(history), {
        "best_val_loss": float(best_val_loss),
        "val_f1": val_eval["f1"],
        "test_f1": test_eval["f1"],
        "test_cm": test_eval["cm"],
    }

def set_global_seed(seed=SEED):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
pca_nn = PCA(n_components=PCA_DIM, random_state=SEED).fit(X_train_dense)
ica_nn = FastICA(
    n_components=ICA_DIM,
    random_state=SEED,
    whiten="unit-variance",
    max_iter=1000,
    tol=1e-4,
).fit(X_train_dense)
rp_nn = GaussianRandomProjection(n_components=RP_DIM, random_state=SEED).fit(X_train_dense)

adult_nn_spaces = {
    "original": {
        "X_train": X_train_dense,
        "X_val": X_val_dense,
        "X_test": X_test_dense,
    },
    "pca": {
        "X_train": pca_nn.transform(X_train_dense),
        "X_val": pca_nn.transform(X_val_dense),
        "X_test": pca_nn.transform(X_test_dense),
    },
    "ica": {
        "X_train": ica_nn.transform(X_train_dense),
        "X_val": ica_nn.transform(X_val_dense),
        "X_test": ica_nn.transform(X_test_dense),
    },
    "rp": {
        "X_train": rp_nn.transform(X_train_dense),
        "X_val": rp_nn.transform(X_val_dense),
        "X_test": rp_nn.transform(X_test_dense),
    },
}

In [ ]:
adult_nn_results = []
adult_nn_histories = {}

for name, data_dict in adult_nn_spaces.items():
    model, hist, metrics = train_mlp_binary(
        data_dict["X_train"], y_train,
        data_dict["X_val"], y_val,
        data_dict["X_test"], y_test,
        hidden_dim=128,
        lr=1e-3,
        batch_size=64,
        max_epochs=60,
        patience=8,
        seed=SEED,
    )
    adult_nn_histories[name] = hist
    adult_nn_results.append(
        {
            "representation": name,
            "n_features": data_dict["X_train"].shape[1],
            **metrics,
        }
    )

adult_nn_results = pd.DataFrame(adult_nn_results).sort_values("test_f1", ascending=False).reset_index(drop=True)
display(adult_nn_results)


In [ ]:
plt.figure()
for name, hist in adult_nn_histories.items():
    plt.plot(hist["epoch"], hist["val_f1"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation F1")
plt.title("Adult - validation F1 by representation")
plt.legend()
plt.show()

## Step 5 (Neural networks with cluster-derived features)

In [ ]:
kmeans_feat = KMeans(
    n_clusters=BEST_KMEANS_K,
    random_state=SEED,
    n_init=20,
    max_iter=500,
).fit(X_train_dense)

gmm_feat = GaussianMixture(
    n_components=BEST_GMM_K,
    covariance_type="diag",
    n_init=3,
    max_iter=300,
    random_state=SEED,
    reg_covar=1e-3,
).fit(X_train_dense)

def build_cluster_feature_sets(X_train_np, X_val_np, X_test_np):
    km_train = kmeans_feat.transform(X_train_np)
    km_val = kmeans_feat.transform(X_val_np)
    km_test = kmeans_feat.transform(X_test_np)

    gmm_train = gmm_feat.predict_proba(X_train_np)
    gmm_val = gmm_feat.predict_proba(X_val_np)
    gmm_test = gmm_feat.predict_proba(X_test_np)

    return {
        "kmeans_only": (km_train, km_val, km_test),
        "gmm_only": (gmm_train, gmm_val, gmm_test),
        "original_plus_kmeans": (
            np.hstack([X_train_np, km_train]),
            np.hstack([X_val_np, km_val]),
            np.hstack([X_test_np, km_test]),
        ),
        "original_plus_gmm": (
            np.hstack([X_train_np, gmm_train]),
            np.hstack([X_val_np, gmm_val]),
            np.hstack([X_test_np, gmm_test]),
        ),
    }

adult_cluster_feature_sets = build_cluster_feature_sets(X_train_dense, X_val_dense, X_test_dense)

In [ ]:
adult_cluster_nn_results = []

for name, (Xt, Xv, Xte) in adult_cluster_feature_sets.items():
    _, hist, metrics = train_mlp_binary(
        Xt, y_train,
        Xv, y_val,
        Xte, y_test,
        hidden_dim=128,
        lr=1e-3,
        batch_size=64,
        max_epochs=60,
        patience=8,
        seed=SEED,
    )
    adult_cluster_nn_results.append(
        {
            "feature_set": name,
            "n_features": Xt.shape[1],
            **metrics,
        }
    )

adult_cluster_nn_results = pd.DataFrame(adult_cluster_nn_results).sort_values("test_f1", ascending=False).reset_index(drop=True)
display(adult_cluster_nn_results)

In [ ]:
adult_step45_summary = pd.concat(
    [
        adult_nn_results.assign(experiment_group="dimensionality_reduction").rename(columns={"representation": "configuration"}),
        adult_cluster_nn_results.assign(experiment_group="cluster_features").rename(columns={"feature_set": "configuration"}),
    ],
    ignore_index=True,
    sort=False,
)
display(adult_step45_summary[["experiment_group", "configuration", "n_features", "best_val_loss", "val_f1", "test_f1"]].sort_values(["experiment_group", "test_f1"], ascending=[True, False]))


## Summary tables

In [ ]:
adult_step1_summary = raw_summary.copy()
adult_step2_summary = pd.concat([pca_results, ica_results, rp_results], ignore_index=True, sort=False)
adult_step3_summary = cluster_comparison_table.copy()

print("Adult Step 1 summary")
display(adult_step1_summary)

print("Adult Step 2 summary")
display(adult_step2_summary.head(20))

print("Adult Step 3 summary")
display(adult_step3_summary)

print("Adult Step 4-5 summary")
display(adult_step45_summary[["experiment_group", "configuration", "n_features", "best_val_loss", "val_f1", "test_f1"]])
